Firstly, let's import every package we will need in this analysis. It is required to install 
```Python
%pip install atlasopenmagic
```
in order to open ATLAS Open Data

In [230]:
import numpy as np
import pandas as pd                 
import uproot                       # to open .root files
import awkward as ak                # to read data with uproot
import matplotlib.pyplot as plt     # to plot
import vector                       # allows to manipulate Lorentz vectors
import os                           # to manage directories
import random                       # extract random numbers
import requests                     # for HTTP access
import aiohttp                      # HTTP client support
import atlasopenmagic as atom       # to access ATLAS Open Data directly

## Import Data
Now we can download the dataset, selecting the release of interest. It is possible to see every release using
```Python
atom.available_releases()
```
We will use the 2025 release of data taken at $\sqrt{s}=13$ TeV in p-p collisions


In [193]:
atom.set_release("2025e-13tev-beta")    # select release
skim = "GamGam"                         # select skim: events with 2 photons
random.seed(24)                         # set seed for random extractions

# get keys for every dataset: Run2 datas have key="data", MC simulations have key=numbers
all_keys = atom.available_datasets()

run_url_list = atom.get_urls("data", skim, protocol="https", cache=True)      # get list of urls for Run2 data

# get list of urls for MC simulations
mc_url_list = []
for key in all_keys:
    if key != "data":
        mc_url_list += atom.get_urls(key, skim, protocol="https", cache=True)


print(f"Number of MonteCarlo simulated datasets: {len(mc_url_list)}")
print(f"Number of Run2 datasets: {len(run_url_list)}")

Release '2025e-13tev-beta' already active with cached metadata.
Active release: 2025e-13tev-beta. (Datasets path: REMOTE)


Number of MonteCarlo simulated datasets: 373
Number of Run2 datasets: 16


MC simulated datasets are produced considering only one process in each dataset. To separate signal datasets from background datasets we have to select urls that contain the Higgs boson decay channel of interest: into two photons.
Its notation in *gamgam* or *yy*.

In [159]:
# select signal datasets: may contain "gamgam" or "yy"
signal_url_list = [url for url in mc_url_list if ("gamgam" in url) or ("Hyy" in url) or (("_yy" in url) and ("yyy" not in url))]

# select background datasets: all dataset not taken in signal
bkg_url_list = [url for url in mc_url_list if url not in signal_url_list]

print("Number of Signal datasets: ", len(signal_url_list))
print("Number of Background datasets: ", len(bkg_url_list))

Number of Signal datasets:  20
Number of Background datasets:  353


Now we open data in TTrees. In order to have a way to train rapidly the model, it is possible to switch between a subset of data and the entire dataset with a boolean `bool useall`

In [160]:
useall = False      # True: use all dataset
                    # False: use a subset

Explore content of datasets: TTree and TBranches

In [161]:
# See names of trees and branches
print("Tree name: ", uproot.open(mc_url_list[0]).keys())
print("Branches: ", uproot.open(f"{mc_url_list[0]}:analysis").keys())

Tree name:  ['analysis;1']
Branches:  ['sig_ph', 'n_sig_ph', 'num_events', 'sum_of_weights', 'sum_of_weights_squared', 'xsec', 'kfac', 'filteff', 'TriggerMatch_DILEPTON', 'ScaleFactor_MLTRIGGER', 'ScaleFactor_PILEUP', 'ScaleFactor_FTAG', 'mcWeight', 'channelNumber', 'eventNumber', 'runNumber', 'trigML', 'trigP', 'trigDT', 'trigT', 'trigE', 'trigDM', 'trigDE', 'trigM', 'trigMET', 'ScaleFactor_BTAG', 'ScaleFactor_JVT', 'jet_n', 'jet_pt', 'jet_eta', 'jet_phi', 'jet_e', 'jet_btag_quantile', 'jet_jvt', 'largeRJet_n', 'largeRJet_pt', 'largeRJet_eta', 'largeRJet_phi', 'largeRJet_e', 'largeRJet_m', 'largeRJet_D2', 'jet_pt_jer1', 'jet_pt_jer2', 'ScaleFactor_ELE', 'ScaleFactor_MUON', 'ScaleFactor_LepTRIGGER', 'ScaleFactor_MuTRIGGER', 'ScaleFactor_ElTRIGGER', 'lep_n', 'lep_type', 'lep_pt', 'lep_eta', 'lep_phi', 'lep_e', 'lep_charge', 'lep_ptvarcone30', 'lep_topoetcone20', 'lep_z0', 'lep_d0', 'lep_d0sig', 'lep_isTightID', 'lep_isMediumID', 'lep_isLooseID', 'lep_isTightIso', 'lep_isLooseIso', 'lep_

Define TTree name and TBranches that we want to extract for this analysis

In [162]:
tree = "analysis"           # name of TTree in ATLAS OD

features = [branch for branch in (uproot.open(f"{mc_url_list[0]}:{tree}").keys()) 
            if ("photon_" in branch) and ("truth" not in branch)]

#if not useall:
#        mc_ind = set(random.sample(range(len(mc_url_list)), 16))
#        mc_url_list = [url for i, url in enumerate(mc_url_list) if i in mc_ind]
        


Read datasets into awkward arrays. This procedure avoids errors in calling CERN server and it is the faster solution.
Then, label=1 is assigned to signal events, label=0 is assigned to background events.

**This download may last up to 15-20 minutes!**
Then, it is possible to save locally the awkward arrays and import them rapidly, without download

In [ ]:
# read signal datasets into awkward array
signal_array = []
for url in signal_url_list:    
    with uproot.open(f"{url}:{tree}") as tree_opened:
        arr = tree_opened.arrays(filter_name=features, library="ak")
        signal_array.append(arr)
signal_awk = ak.concatenate(signal_array)

# assign label 1 to signal
signal_awk["label"] = 1


# read background datasets into awkward array
bkg_array = []
for url in bkg_url_list:
    with uproot.open(f"{url}:{tree}") as tree_opened:
        arr = tree_opened.arrays(filter_name=features, library="ak")
        bkg_array.append(arr)
bkg_awk = ak.concatenate(bkg_array)

# assign label 0 to background
bkg_awk["label"] = 0            


# merge signal and background arrays into one
mc_awk = ak.concatenate([signal_awk, bkg_awk])

# create directory if needed and save array locally
os.makedirs("data", exist_ok=True)
ak.to_parquet(mc_awk, "data/mc_awkward_complete.parquet") 
print("MC awkward array correctly saved!")

Awkward array correctly saved!


In [ ]:
# recover mc and run2 awkward arrays from local memory
mc_awk = ak.from_parquet("data/mc_awkward_complete.parquet")

print("MC awkward arrays correctly recovered!")

Awkward array correctly recovered!


## Preprocessing Dataset
Now that we have the complete dataset, we need to preprocess for the Neural Network input.

Firstly, for each event, we order photons for decreasing $p_T$ in order to select the **leading** (index 0) and **subleading** (index 1) photons. These are the photons of interest for this analysis.

To do so, we need to separate *jagged* variables, that are associated to each photon, from *scalar* variables, that refers to the entire event

In [212]:
# define function that sorts photons by pT: we will also use it for run2 dataset
def sort_photons_pt(array):
    # separate jagged and scalar var
    scalar_var = ["photon_n", "label"]
    jagged_var = [var for var in features if var not in scalar_var]

    # find indexes of ordered photon, from higher pT to lower pT
    sorted_index = ak.argsort(array.photon_pt, axis=-1, ascending=False)
    
    # sort mc array with dictionary unpacking
    return ak.Array({
        **{var: array[var][sorted_index] for var in jagged_var},
        **{var: array[var] for var in scalar_var},
    })

In [ ]:
# sort MC array
mc_sort = sort_photons_pt(mc_awk)

# check that photons have been ordered
n_events_notord = len(mc_sort[mc_sort["photon_pt",:,0]<mc_sort["photon_pt",:,1]])
if n_events_notord == 0:
    print("Every event has been properly sorted by pT!")
else:
    print(f"Error: there are {n_events_notord} events that are not sorted")

Every event has been properly sorted by pT!


Last check: we want to select only photons that are well reconstructed and isolated. To do so, we impose that leading and subleading photons respect the 4 boolean conditions in the dataset.

In [214]:
# define a function that selects only well reconstructed and isolated photons
def select_reconst_and_isolated_phtons(array):
    # make mask for those conditions
    mask = (
        array.photon_isLooseID[:, 0] & array.photon_isLooseID[:, 1] &
        array.photon_isTightID[:, 0] & array.photon_isTightID[:, 1] &
        array.photon_isLooseIso[:, 0] & array.photon_isLooseIso[:, 1] &
        array.photon_isTightIso[:, 0] & array.photon_isTightIso[:, 1]
    )
    # apply mask to array
    return array[mask]

In [ ]:
# filter MC dataset: select only events with well reconstructed and isolated photons
mc_filtered = select_reconst_and_isolated_phtons(mc_sort)

### Invariant Mass $m_{\gamma\gamma}$
Now it is useful to calculate the invariant mass of leading and subleading photons for each event. In natural units, the invariant mass is defined as
$$ m_{\gamma\gamma} = \sqrt{E^2_\text{tot}-(\vec{p}_\text{tot})^2} $$
We can calculate it using `.M` methon in `vector` library, using 4-momentum-like vectors defined as 
$$ p_4 = (E, p_T, \phi, \eta) $$
The result is given in GeV

In [232]:
# define a function to calculate invariant mass
def invariant_mass_calc(array):
    # build 4-momentum for each photon
    p4 = vector.zip({
        "pt": array.photon_pt,
        "eta": array.photon_eta,
        "phi": array.photon_phi,
        "e": array.photon_e
    })
    return (p4[:, 0] + p4[:, 1]).M

In [263]:
# add invariant mass column in MC awkward array 
mc_filtered["photon_invariant_mass"] = invariant_mass_calc(mc_filtered)

# remove events with invariant mass = 0, as later we have to divide for it
mc_final = mc_filtered[mc_filtered["photon_invariant_mass"]!=0]

## Features Extraction
Now that we have everything we need, let's proceed to extract features with some precautions.

- $p_T$: in order to avoid the NN to learn $m_{\gamma\gamma}$ dependence, we normalize $p_T$ for it
- $E_\gamma$: same procedure
- $\eta$: is already adimensional and $m_{\gamma\gamma}-independent
- $\phi$: to avoid differences of $-\pi$ and $+\pi$, we transform it into two features: $\cos\phi$ and $\sin\phi$
- ptcone20: for same reasons as $p_T$, normalize wrt $p_T$ to get relative isolation
- topoetcone40: same as previous

In [245]:
# define a function that separates features for leading and subleading photon
def get_features(array):
    """
    Input: awkward array, that is the filtered dataset
    Output: two awkward arrays containing features, first for leading and second for subleading
    """
    # get leading photon features
    lead_features = {
        "pt": array.photon_pt[:, 0] / array.photon_invariant_mass,
        "eta": array.photon_eta[:, 0],
        "cos_phi": np.cos(array.photon_phi[:, 0]),
        "sin_phi": np.sin(array.photon_phi[:, 0]),
        "e": array.photon_e[:, 0] / array.photon_invariant_mass,
        "ptcone20": array.photon_ptcone20[:, 0] / array.photon_pt[:, 0],
        "topoetcone40": array.photon_topoetcone40[:, 0] / array.photon_pt[:, 0]
    }

    # get subleading photon features
    sublead_features = {
            "pt": array.photon_pt[:, 1] / array.photon_invariant_mass,
            "eta": array.photon_eta[:, 1],
            "cos_phi": np.cos(array.photon_phi[:, 1]),
            "sin_phi": np.sin(array.photon_phi[:, 1]),
            "e": array.photon_e[:, 1] / array.photon_invariant_mass,
            "ptcone20": array.photon_ptcone20[:, 1] / array.photon_pt[:, 1],
            "topoetcone40": array.photon_topoetcone40[:, 1] / array.photon_pt[:, 1]
        }

    # transform dictionaries into awkward arrays
    return ak.Array(lead_features), ak.Array(sublead_features)

In [262]:
# obtain features for MC dataset
leadig_photon_features, subleading_photon_features = get_features(mc_final)

In [259]:
# preview

mc_awk[mc_awk["photon_pt",:,0]<mc_awk["photon_pt",:,1]]
mc_awk[mc_awk["photon_n"]>3]
ak.to_dataframe(mc_filtered[mc_filtered["photon_invariant_mass"]==0])

photon_pt  photon_eta  photon_phi   photon_e  photon_ptcone20  \
entry subentry                                                                  
0     0         27.300594   -0.439359    1.501697  29.978256              0.0   
      1         25.503351   -0.439617    1.501697  28.007732              0.0   
1     0         29.005808   -0.478821    1.550526  32.394897              0.0   
      1         27.301413   -0.478662    1.550526  30.489201              0.0   
2     0         38.339340   -0.225342   -1.157462  39.316887              0.0   
      1         26.539707   -0.225143   -1.157462  27.215191              0.0   
3     0         28.029453   -0.417788   -1.660275  30.511471              0.0   
      1         27.387468   -0.418391   -1.660275  29.819748              0.0   
4     0         37.523315    0.883785    0.975810  53.156620              0.0   
      1         33.340000    0.884095    0.975810  47.240780              0.0   
5     0         28.954380    0.625217    1.754526  34.800217              0.0   
      1         27.994678    0.625295    1.754526  33.648220              0.0   
6     0         30.241030    0.450844   -1.206798  33.366848              0.0   
      1         26.918755    0.450724   -1.206798  29.699659              0.0   
7     0         32.672279   -0.181252    0.625111  33.210426              0.0   
      1         30.810837   -0.181740    0.625111  31.321070              0.0   
8     0         40.811615   -0.763343    0.704842  53.290627              0.0   
      1         34.343246   -0.763489    0.704842  44.848633              0.0   
9     0         29.249653   -0.118920   -1.850704  29.456720              0.0   
      1         27.880474   -0.118584   -1.850704  28.076735              0.0   
10    0         27.055344   -0.335003   -2.346878  28.587767              0.0   
      1         25.120291   -0.334243   -2.346878  26.536598              0.0   
11    0         25.959061    1.172273    0.412287  45.934372              0.0   
      1         25.517946    1.171098    0.412287  45.110077              0.0   
12    0         25.980078    0.282739   -0.197194  27.025457              0.0   
      1         25.318424    0.283027   -0.197194  26.339266              0.0   
13    0         30.370249    0.188882    1.879554  30.913609              0.0   
      1         27.681334    0.188231    1.879554  28.173174              0.0   
14    0         29.837831   -0.182965   -1.449014  30.338657              0.0   
      1         29.508312   -0.182719   -1.449014  30.002268              0.0   
15    0         30.659405   -0.657912    1.502466  37.537655              0.0   
      1         28.853289   -0.658375    1.502466  35.335789              0.0   
16    0         36.489723   -0.768057    2.669113  47.792171              0.0   
      1         32.452271   -0.768392    2.669113  42.513367              0.0   
17    0         38.276066   -0.529818   -0.562480  43.775108              0.0   
      1         25.090015   -0.529815   -0.562480  28.694603              0.0   
18    0         28.642986   -0.469736    1.266844  31.861589              0.0   
      1         27.748295   -0.468882    1.266844  30.854818              0.0   

                photon_topoetcone40  photon_isLooseID  photon_isTightID  \
entry subentry                                                            
0     0                    0.025831              True              True   
      1                    0.473070              True              True   
1     0                    1.300532              True              True   
      1                    1.007597              True              True   
2     0                   -2.214072              True              True   
      1                   -2.004708              True              True   
3     0                   -1.130424              True              True   
      1                   -2.229401              True              True   
4     0                  